# Reclamações 04 · Análise

## 0. Contexto

Este notebook responde às cinco perguntas de negócio sobre reclamações, na ordem de uma narrativa: do que o cliente reclama, quem reduziu as reclamações, se a procedência conta toda a história, se técnico e comercial andam juntos e se as reclamações técnicas confirmam o ranking de continuidade.

Cada pergunta segue a mesma estrutura: o que será analisado e como ler o gráfico, o gráfico, a conclusão do gráfico e, ao final, a conclusão da pergunta. As tabelas de apoio usadas durante a análise estão comentadas no código; basta retirar o comentário para exibi-las.

| # | Pergunta | Seção |
|---|---|---|
| P1 | Quais grupos de reclamação comercial mais pesam para o consumidor e quais mais contribuíram para a variação? | 1 e 2 |
| P2 | Quais distribuidoras mais reduziram as reclamações comerciais recebidas e procedentes por mil UCs, e em quais grupos essa redução ocorreu? | 2 |
| P3 | A redução das procedentes, se houve, vem acompanhada de redução das recebidas, ou pode refletir maior rigor na classificação de procedência? | 3 |
| P4 | As reclamações técnicas e comerciais evoluem juntas em cada distribuidora? | 4 |
| P5 | O ranking de reclamações de Qualidade confirma o ranking de continuidade (DEC-FI e FEC-FI) na mesma janela? | 5 |

**Universo:** as 32 distribuidoras de grande porte, sem a CELESC, excluída por série internamente inconsistente (`02_silver_complaints`, seção 11).

### Limites declarados

1. O indicador segue a regra de exclusão do FER (PRODIST Módulo 8, item 285), mas não é o FER oficial: o denominador é a média mensal de UCs na janela, e não o número de consumidores de dezembro. Os valores não são comparáveis a FER publicados.
2. O recorte comercial estrito é decisão deste trabalho: retira do item 285 Rede/Manutenção e Outros de Qualidade, que fariam o indicador medir causa técnica.
3. O grupo Qualidade inclui interrupção programada, tensão e outros de qualidade, 2,4% do grupo em 2024.
4. Pagamento e Outras comerciais têm volume baixo por distribuidora, assim como a Geração distribuída nas distribuidoras menores; a variação percentual é mais sensível a oscilação, e o volume absoluto acompanha a posição.
5. O indicador de Geração distribuída usa o total de UCs no denominador, e não as unidades com geração distribuída, que não estão no pipeline. A variação mistura, portanto, o crescimento da adesão à geração distribuída e a qualidade do atendimento a esses clientes.
6. Os resultados são descritivos. Só as correlações trazem valor-p; a inferência sobre a série mensal fica como trabalho futuro.

## Configuração

In [ ]:
import os
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.patches import Patch, Rectangle
from matplotlib.transforms import Bbox
from pyspark.sql import functions as F
from scipy.stats import spearmanr

# Walk up from the working directory until the folder holding `src` is found,
# so the notebook works at any depth inside notebooks/
REPO_ROOT = os.getcwd()
while not os.path.isdir(os.path.join(REPO_ROOT, "src")):
    parent = os.path.dirname(REPO_ROOT)
    if parent == REPO_ROOT:
        raise FileNotFoundError("Repository root with a src folder not found above " + os.getcwd())
    REPO_ROOT = parent
if REPO_ROOT not in sys.path:
    sys.path.append(REPO_ROOT)

from src.config import CATALOG, SCHEMA_GOLD, SCHEMA_SILVER

SILVER = f"{CATALOG}.{SCHEMA_SILVER}"
GOLD = f"{CATALOG}.{SCHEMA_GOLD}"

RECORTE_TOTAL = "comercial_estrito"
GRUPOS_COMERCIAIS = ["faturamento", "pagamento", "geracao_distribuida", "outras_comerciais"]
ROTULO_GRUPO = {"faturamento": "Faturamento", "pagamento": "Pagamento",
                "geracao_distribuida": "Geração distribuída",
                "outras_comerciais": "Outras comerciais", "qualidade": "Qualidade"}
MESES = {1: "jan", 2: "fev", 3: "mar", 4: "abr", 5: "mai", 6: "jun",
         7: "jul", 8: "ago", 9: "set", 10: "out", 11: "nov", 12: "dez"}

# Palette validated for color-vision deficiency: one hue per commercial group,
# a blue/orange pair for improvement/worsening and grays for context
COR_GRUPO = {"faturamento": "#4a3aa7", "pagamento": "#eda100",
             "geracao_distribuida": "#e87ba4", "outras_comerciais": "#008300"}
# Text color that stays readable inside each group's segment
COR_ROTULO = {"faturamento": "white", "pagamento": "#2b2b29",
              "geracao_distribuida": "#2b2b29", "outras_comerciais": "white"}
COR_MELHORA, COR_PIORA = "#2a78d6", "#eb6834"
COR_NEUTRA, COR_TEXTO, COR_EIXO = "#b4b2ab", "#52514e", "#8a8983"
COR_ALERTA, COR_PONTO, COR_NOME = "#c62828", "#8a8983", "#9e9d98"
FUNDO_BOM, FUNDO_ALERTA, FUNDO_PIORA, FUNDO_NEUTRO = "#eaf2fb", "#fde8e4", "#fdf1e6", "#f4f3ef"

# Clean look for a non-technical audience: no default grid, no top/right frame
plt.rcParams.update({
    "figure.dpi": 110, "font.size": 9,
    "axes.titlesize": 11, "axes.titleweight": "bold", "axes.titlelocation": "left",
    "axes.labelsize": 9, "axes.labelcolor": COR_TEXTO, "axes.edgecolor": "#c8c7c1",
    "axes.spines.top": False, "axes.spines.right": False, "axes.grid": False,
    "xtick.color": COR_TEXTO, "ytick.color": COR_TEXTO, "xtick.labelsize": 8, "ytick.labelsize": 8,
    "legend.frameon": False, "legend.fontsize": 8, "axes.titlepad": 12,
})


def rotulo(ano_mes):
    """Short Portuguese label for a yyyymm window end, e.g. 202412 -> dez/24."""
    return f"{MESES[ano_mes % 100]}/{str(ano_mes // 100)[2:]}"


def br(v, casas=2):
    """Number with Portuguese decimal comma for chart titles."""
    return f"{v:.{casas}f}".replace(".", ",")


def grade_leve(ax, eixo="x"):
    """Very light reference grid on a single axis."""
    ax.grid(True, axis=eixo, color="#ecebe7", linewidth=0.8)
    ax.set_axisbelow(True)


def faixa_eixo(valores):
    """Axis range covering every point except the far ones (boxplot rule), always including zero."""
    v = np.asarray(valores, dtype=float)
    q1, q3 = np.percentile(v, [25, 75])
    iqr = q3 - q1
    dentro = v[(v >= q1 - 1.5 * iqr) & (v <= q3 + 1.5 * iqr)]
    lo, hi = min(dentro.min(), 0.0), max(dentro.max(), 0.0)
    folga = (hi - lo) * 0.08
    return lo - folga, hi + folga


def rotular_pontos(ax, xs, ys, textos, destaque, cores):
    """Name every point without overlaps: highlighted names in the point color, the others small and gray.

    Greedy placement: for each label, try positions around its point and keep the first one
    that overlaps neither another label nor another point.
    """
    fig = ax.figure
    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()
    area = ax.get_window_extent(renderer)
    pixels = ax.transData.transform(np.column_stack([xs.values, ys.values]))
    pontos = {dx: Bbox.from_bounds(px - 4, py - 4, 8, 8) for dx, (px, py) in zip(xs.index, pixels)}
    ocupado = []
    candidatos = [(6, 0, "left", "center"), (-6, 0, "right", "center"), (0, 6, "center", "bottom"),
                  (0, -6, "center", "top"), (5, 5, "left", "bottom"), (-5, 5, "right", "bottom"),
                  (5, -5, "left", "top"), (-5, -5, "right", "top"), (12, 0, "left", "center"),
                  (-12, 0, "right", "center"), (0, 13, "center", "bottom"), (0, -13, "center", "top")]
    ordem = [dx for dx in xs.index if dx in destaque] + [dx for dx in xs.index if dx not in destaque]
    for dx in ordem:
        estilo = (dict(fontsize=7.5, color=cores[dx]) if dx in destaque
                  else dict(fontsize=6.5, color=COR_NOME))
        outros = [b for k, b in pontos.items() if k != dx]
        escolhido = None
        for ox, oy, ha, va in candidatos:
            t = ax.annotate(textos[dx], (xs[dx], ys[dx]), xytext=(ox, oy), textcoords="offset points",
                            ha=ha, va=va, zorder=4, annotation_clip=False, **estilo)
            caixa = t.get_window_extent(renderer)
            livre = not any(caixa.overlaps(b) for b in ocupado + outros)
            dentro = area.x0 <= caixa.x0 and caixa.x1 <= area.x1 and area.y0 <= caixa.y0 and caixa.y1 <= area.y1
            if livre and dentro:
                escolhido = caixa
                break
            t.remove()
        if escolhido is None:
            # No free spot: place the label toward the inside of the chart
            direita = pixels[list(xs.index).index(dx)][0] > (area.x0 + area.x1) / 2
            t = ax.annotate(textos[dx], (xs[dx], ys[dx]), xytext=(-6 if direita else 6, 0),
                            textcoords="offset points", ha="right" if direita else "left", va="center",
                            zorder=4, annotation_clip=False, **estilo)
            escolhido = t.get_window_extent(renderer)
        ocupado.append(escolhido)


def matriz_quadrantes(ax, x, y, cores, destaque, fundos, unidade_x="%", unidade_y="%"):
    """Four-quadrant scatter cut at zero, with every company named.

    Far points are pinned to the border as diamonds and labeled with their real values;
    `fundos` maps (x >= 0, y >= 0) to the background tint of each quadrant.
    """
    (x0, x1), (y0, y1) = faixa_eixo(x), faixa_eixo(y)
    ax.set_xlim(x0, x1)
    ax.set_ylim(y0, y1)
    for (xp, yp), cor_fundo in fundos.items():
        ax.add_patch(Rectangle((0 if xp else x0, 0 if yp else y0),
                               (x1 if xp else 0) - (0 if xp else x0),
                               (y1 if yp else 0) - (0 if yp else y0),
                               facecolor=cor_fundo, edgecolor="none", zorder=0))
    ax.axvline(0, color=COR_EIXO, linewidth=0.8, zorder=1)
    ax.axhline(0, color=COR_EIXO, linewidth=0.8, zorder=1)
    xs, ys = x.clip(x0, x1), y.clip(y0, y1)
    fora = (xs != x) | (ys != y)
    ax.scatter(xs[~fora], ys[~fora], c=cores[~fora].tolist(), s=38, edgecolor="white", linewidth=0.8, zorder=3)
    ax.scatter(xs[fora], ys[fora], c=cores[fora].tolist(), s=60, marker="D", edgecolor="white", linewidth=0.8,
               zorder=3, clip_on=False)
    textos = {dx: (f"{dx} ({x[dx]:+.0f}{unidade_x}; {y[dx]:+.0f}{unidade_y})" if fora[dx] else dx)
              for dx in x.index}
    rotular_pontos(ax, xs, ys, textos, set(destaque) | set(fora[fora].index), cores)


def esquema_quadrantes(titulo, eixo_x, eixo_y, textos, fundos):
    """Explanatory 2x2 diagram: one caption per quadrant, keyed by (x >= 0, y >= 0)."""
    fig, ax = plt.subplots(figsize=(7.5, 4.2))
    for (xp, yp), texto in textos.items():
        x0, y0 = (0.5 if xp else 0.0), (0.5 if yp else 0.0)
        ax.add_patch(Rectangle((x0, y0), 0.5, 0.5, facecolor=fundos[(xp, yp)],
                               edgecolor="white", linewidth=4))
        ax.text(x0 + 0.25, y0 + 0.25, texto, ha="center", va="center", fontsize=9.5, color="#2b2b29")
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_xticks([0.25, 0.75], ["em queda", "em alta"])
    ax.set_yticks([0.25, 0.75], ["em queda", "em alta"])
    ax.tick_params(length=0, labelsize=9)
    for lado in ax.spines.values():
        lado.set_visible(False)
    ax.set_xlabel(eixo_x, fontsize=10)
    ax.set_ylabel(eixo_y, fontsize=10)
    ax.set_title(titulo)
    plt.tight_layout()
    plt.show()


def barras_variacao(serie, titulo, rotulo_x):
    """Horizontal ranking bars: blue for reduction, orange for increase, value inside the bar."""
    fig, ax = plt.subplots(figsize=(9, max(5, len(serie) * 0.26)))
    cores = [COR_MELHORA if v < 0 else COR_PIORA for v in serie]
    ax.barh(serie.index[::-1], serie[::-1], color=cores[::-1], height=0.75)
    amplitude = serie.max() - serie.min()
    for i, v in enumerate(serie[::-1]):
        texto = f"{v:+.0f}%"
        if abs(v) >= amplitude * 0.07:
            # Long enough: the value sits inside the bar, next to its end
            ax.text(v / 2 if abs(v) < amplitude * 0.12 else v - np.sign(v) * amplitude * 0.01, i, texto,
                    va="center", ha="center" if abs(v) < amplitude * 0.12 else ("right" if v > 0 else "left"),
                    fontsize=7, color="white")
        else:
            # Too short: the value goes just outside the bar end
            ax.text(v + (amplitude * 0.01 if v >= 0 else -amplitude * 0.01), i, texto, va="center",
                    ha="left" if v >= 0 else "right", fontsize=7, color=COR_TEXTO)
    ax.axvline(0, color=COR_EIXO, linewidth=0.8)
    ax.set_xlim(min(serie.min(), 0) - amplitude * 0.06, max(serie.max(), 0) + amplitude * 0.06)
    ax.set_xticks([])
    ax.spines["bottom"].set_visible(False)
    ax.tick_params(axis="y", length=0)
    ax.set_xlabel(rotulo_x)
    ax.set_title(titulo)
    plt.tight_layout()
    plt.show()


nomes = (spark.table(f"{SILVER}.dim_distribuidora")
         .select("num_cnpj", F.trim("sig_agente").alias("sig_agente")).toPandas())

janela = spark.table(f"{GOLD}.fato_reclamacao_janela").toPandas().merge(nomes, on="num_cnpj")
ranking = spark.table(f"{GOLD}.ranking_reclamacoes").toPandas().merge(nomes, on="num_cnpj")
ranking_cont = spark.table(f"{GOLD}.ranking_continuidade").toPandas().merge(nomes, on="num_cnpj")

FINS = sorted(janela["fim_janela"].unique())
INI, FIM = FINS[0], FINS[-1]
univ = janela[~janela["excluida_ranking"] & janela["janela_valida"]].copy()


def rk(recorte, medida):
    """One ranking as a pandas frame indexed by company."""
    return (ranking[(ranking["recorte"] == recorte) & (ranking["medida"] == medida)]
            .set_index("sig_agente").sort_values("posicao"))


def valor(recorte, fim, coluna):
    """One indicator of one window, indexed by company."""
    return univ[(univ["recorte"] == recorte) & (univ["fim_janela"] == fim)].set_index("sig_agente")[coluna]


MEMBROS = list(rk(RECORTE_TOTAL, "procedentes").index)


def indicador_universo(recorte, fim, qtd="qtd_procedentes"):
    """Indicator of the whole universe: total complaints over total consumer units, per thousand."""
    t = univ[(univ["recorte"] == recorte) & (univ["fim_janela"] == fim) & univ["sig_agente"].isin(MEMBROS)]
    return t[qtd].sum() / t["ucs_media"].sum() * 1000


print(f"Janelas.........: {[rotulo(f) for f in FINS]}")
print(f"Comparacao......: {rotulo(INI)} contra {rotulo(FIM)}")
print(f"Distribuidoras..: {len(MEMBROS)} no universo analisado")

## 1. Do que o cliente reclama (P1, primeira parte)

Antes de perguntar quem melhorou, é preciso saber do que o cliente reclama. Esta seção mostra o peso de cada tipo de reclamação e como esse peso evoluiu.

#### Como ler os números

- **Nível 1:** reclamação registrada no atendimento da própria distribuidora. A ouvidoria (nível 2) recebe reclamações que já passaram pelo nível 1, e somar os dois contaria o mesmo problema duas vezes; por isso ela aparece à parte, na seção 3.
- **Recebida:** toda reclamação registrada. **Procedente:** reclamação que a própria distribuidora reconheceu como fundamentada.
- **LTM (Last Twelve Months, os últimos 12 meses):** soma dos 12 meses encerrados no mês indicado. "LTM jun/26" vai de julho de 2025 a junho de 2026.

#### Técnico e comercial

A família de reclamações da REH 2.992/2021 mistura dois mundos: o técnico, dominado pela falta de energia, e o comercial, que trata de fatura, pagamento, ligação e atendimento. O gráfico mostra o peso de cada um na janela mais recente.

In [ ]:
fato = spark.table(f"{SILVER}.fato_manifestacao")
tipologia = spark.table(f"{SILVER}.dim_tipologia")
universo_spark = spark.createDataFrame(univ[univ["sig_agente"].isin(MEMBROS)][["num_cnpj"]].drop_duplicates())

# The twelve months of the last window, e.g. 202507..202606
meses_fim = [((FIM // 100) - (1 if m > FIM % 100 else 0)) * 100 + m for m in range(1, 13)]
fato_fim = (fato.filter((F.col("nivel") == 1) & F.col("ano_mes").isin(meses_fim))
            .join(universo_spark, "num_cnpj"))

ROTULO_BLOCO = {"tecnico": "Técnico: interrupção, tensão\ne danos elétricos",
                "rede_e_qualidade_outros": "Rede, manutenção e\noutros de qualidade",
                "comercial_estrito": "Comercial"}
COR_BLOCO = {"tecnico": COR_NEUTRA, "rede_e_qualidade_outros": "#d8d6cf", "comercial_estrito": COR_MELHORA}

blocos = (fato_fim
    .join(tipologia.filter(F.col("bloco").isin(list(ROTULO_BLOCO))).select("cod_tipologia", "bloco"),
          "cod_tipologia")
    .groupBy("bloco")
    .agg(F.sum("qtd_recebidas").alias("recebidas"), F.sum("qtd_procedentes").alias("procedentes"))
    .toPandas().set_index("bloco").loc[list(ROTULO_BLOCO)])
participacao_bloco = blocos / blocos.sum() * 100

fig, axes = plt.subplots(1, 2, figsize=(11, 2.8), sharey=True)
for ax, medida in zip(axes, ["recebidas", "procedentes"]):
    valores = participacao_bloco[medida]
    ax.barh([ROTULO_BLOCO[b] for b in valores.index], valores,
            color=[COR_BLOCO[b] for b in valores.index], height=0.6)
    for i, v in enumerate(valores):
        ax.text(v + 1.5, i, f"{v:.1f}%".replace(".", ","), va="center", fontsize=9, color=COR_TEXTO)
    ax.set_xlim(0, 112)
    ax.set_xticks([])
    ax.tick_params(axis="y", length=0)
    ax.spines["bottom"].set_visible(False)
    ax.invert_yaxis()
    ax.set_title(f"Reclamações {medida}")
fig.suptitle(f"Peso do técnico e do comercial nas reclamações, LTM até {rotulo(FIM)}",
             x=0.01, ha="left", fontweight="bold", fontsize=11)
plt.tight_layout()
plt.show()

# display(blocos.assign(**{f"{c}_pct": participacao_bloco[c].round(1) for c in blocos.columns})
#         .rename(index=lambda b: ROTULO_BLOCO[b].replace("\n", " ")).reset_index())

**Conclusão:** o técnico responde por 84,6% das reclamações recebidas e por 90,1% das procedentes; o comercial, por 4,5% e 1,9%. Somados, o volume da falta de energia esconderia qualquer movimento comercial. Por isso as duas frentes são analisadas separadamente: o comercial nas seções 2 e 3, o técnico nas seções 4 e 5.

#### Os grupos de reclamação comercial

Dentro do comercial, a REH organiza as tipologias em grupos, o 2º nível da classificação. À esquerda, as recebidas: do que o cliente reclama. À direita, as procedentes: o que a distribuidora reconheceu. A cor indica o grupo usado nos rankings: Faturamento, Pagamento e Geração distribuída têm ranking próprio, e os demais grupos formam Outras comerciais.

In [ ]:
def encurtar(texto, limite=48):
    """Trim long descriptions for chart labels."""
    texto = " ".join(str(texto).split())
    return texto if len(texto) <= limite else texto[:limite - 3].rstrip() + "..."


por_grupo_reh = (fato_fim
    .join(tipologia.filter("ind_comercial_estrito")
          .select("cod_tipologia", "cod_nivel_2", "desc_nivel_2", "grupo_ranking"), "cod_tipologia")
    .groupBy("cod_nivel_2", "desc_nivel_2", "grupo_ranking")
    .agg(F.sum("qtd_recebidas").alias("recebidas"), F.sum("qtd_procedentes").alias("procedentes"))
    .toPandas())

# Level-2 code 10217 is itself described as "Outros"; make the label self-explanatory
por_grupo_reh["grupo_reh"] = [encurtar("Outros assuntos comerciais" if cod == "10217" else desc)
                              for cod, desc in zip(por_grupo_reh["cod_nivel_2"], por_grupo_reh["desc_nivel_2"])]

fig, axes = plt.subplots(1, 2, figsize=(13, 5.2))
for ax, medida in zip(axes, ["recebidas", "procedentes"]):
    ordenado = por_grupo_reh.sort_values(medida, ascending=False)
    valores = list(ordenado[medida] / ordenado[medida].sum() * 100)
    ax.barh(range(len(valores)), valores, color=[COR_GRUPO[g] for g in ordenado["grupo_ranking"]], height=0.7)
    for i, v in enumerate(valores):
        ax.text(v + 0.6, i, f"{v:.1f}%".replace(".", ","), va="center", fontsize=8, color=COR_TEXTO)
    ax.set_yticks(range(len(valores)), ordenado["grupo_reh"], fontsize=8)
    ax.tick_params(axis="y", length=0)
    ax.invert_yaxis()
    ax.set_xlim(0, max(valores) * 1.18)
    ax.set_xticks([])
    ax.spines["bottom"].set_visible(False)
    ax.set_title(f"Reclamações {medida} (% do comercial estrito)")
fig.legend(handles=[Patch(color=COR_GRUPO[g], label=ROTULO_GRUPO[g]) for g in GRUPOS_COMERCIAIS],
           loc="lower center", ncol=4, bbox_to_anchor=(0.5, -0.02))
fig.suptitle(f"Comercial estrito: reclamações por grupo da REH (2º nível), LTM até {rotulo(FIM)}",
             x=0.01, ha="left", fontweight="bold", fontsize=11)
plt.tight_layout(rect=(0, 0.04, 1, 1))
plt.show()

por_grupo_reh["taxa_procedencia_pct"] = (por_grupo_reh["procedentes"] / por_grupo_reh["recebidas"] * 100).round(1)
# display(por_grupo_reh.sort_values("recebidas", ascending=False)
#         [["cod_nivel_2", "grupo_reh", "grupo_ranking", "recebidas", "procedentes", "taxa_procedencia_pct"]]
#         .reset_index(drop=True))

**Conclusão:** Leitura/Faturamento/Fatura responde por metade do comercial (51,8% das recebidas e 48,9% das procedentes). A Geração distribuída vem em segundo, com 13,5% das recebidas e 17,8% das procedentes, bem à frente de Pagamento (6,3%). A procedência varia muito entre grupos: 40,5% das reclamações de Geração distribuída são procedentes, contra 28,9% no Faturamento; Iluminação Pública chega a 82%, e Outros assuntos comerciais (11,7%) e Procedimento Irregular (14,2%) ficam no outro extremo.

#### Evolução do peso de cada grupo

As barras mostram quanto cada grupo de ranking representa do comercial estrito em cada janela de 12 meses.

In [ ]:
comp = (univ[univ["recorte"].isin(GRUPOS_COMERCIAIS) & univ["sig_agente"].isin(MEMBROS)]
        .groupby(["fim_janela", "recorte"])[["qtd_procedentes", "qtd_recebidas"]].sum()
        .reset_index())

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8), sharey=True)
for ax, medida, titulo in zip(axes, ["qtd_recebidas", "qtd_procedentes"], ["Recebidas", "Procedentes"]):
    tabela = comp.pivot(index="fim_janela", columns="recorte", values=medida)[GRUPOS_COMERCIAIS]
    tabela = tabela.div(tabela.sum(axis=1), axis=0) * 100
    base = np.zeros(len(tabela))
    for grupo in GRUPOS_COMERCIAIS:
        ax.bar([f"LTM {rotulo(f)}" for f in tabela.index], tabela[grupo], bottom=base, width=0.65,
               color=COR_GRUPO[grupo], edgecolor="white", linewidth=1.5, label=ROTULO_GRUPO[grupo])
        for x, (b, v) in enumerate(zip(base, tabela[grupo])):
            if v >= 4:
                ax.text(x, b + v / 2, f"{v:.0f}%", ha="center", va="center", color=COR_ROTULO[grupo], fontsize=8.5)
        base += tabela[grupo].values
    ax.set_yticks([])
    ax.spines["left"].set_visible(False)
    ax.set_title(titulo)
axes[1].legend(loc="upper left", bbox_to_anchor=(1.0, 1.0))
fig.suptitle("Peso de cada grupo no comercial estrito, por janela de 12 meses",
             x=0.01, ha="left", fontweight="bold", fontsize=11)
plt.tight_layout()
plt.show()

comp_gd = comp.pivot(index="fim_janela", columns="recorte", values="qtd_procedentes")
print("participacao da Geracao distribuida nas procedentes, universo:")
print((comp_gd["geracao_distribuida"] / comp_gd[GRUPOS_COMERCIAIS].sum(axis=1) * 100).round(1)
      .rename(index=rotulo).to_string())

**Conclusão:** a Geração distribuída é o grupo que mais cresce: sua participação nas procedentes passou de 12,0% (LTM dez/24) para 17,8% (LTM jun/26). O Faturamento oscila entre 49% e 52%, o Pagamento cai de 8% para 6% e Outras comerciais, de 29% para 27%.

#### Composição por distribuidora

O agregado pode esconder distribuidoras em que outro grupo lidera. Cada barra mostra como as procedentes do comercial estrito de uma distribuidora se dividem entre os grupos, na janela inicial e na final; os números dentro das barras são a participação em %. As distribuidoras estão ordenadas pelo peso do Faturamento na janela final.

In [ ]:
por_dx = univ[univ["recorte"].isin(GRUPOS_COMERCIAIS) & univ["fim_janela"].isin([INI, FIM])
              & univ["sig_agente"].isin(MEMBROS)]
por_dx = por_dx.pivot_table(index=["sig_agente", "fim_janela"], columns="recorte",
                            values="qtd_procedentes", aggfunc="sum")[GRUPOS_COMERCIAIS]
por_dx = por_dx.div(por_dx.sum(axis=1), axis=0) * 100
ordem = por_dx.xs(FIM, level="fim_janela").sort_values("faturamento").index

MINIMO_ROTULO = 7  # segments narrower than this (in %) get no label, the number would not fit

fig, axes = plt.subplots(1, 2, figsize=(13, max(6, len(ordem) * 0.3)), sharey=True)
for ax, fim in zip(axes, [INI, FIM]):
    tabela = por_dx.xs(fim, level="fim_janela").reindex(ordem).fillna(0)
    base = np.zeros(len(tabela))
    for grupo in GRUPOS_COMERCIAIS:
        valores = tabela[grupo].values
        ax.barh(tabela.index, valores, left=base, height=0.78, color=COR_GRUPO[grupo],
                edgecolor="white", linewidth=0.8, label=ROTULO_GRUPO[grupo])
        for i, (b, v) in enumerate(zip(base, valores)):
            if v >= MINIMO_ROTULO:
                ax.text(b + v / 2, i, f"{v:.0f}", ha="center", va="center",
                        fontsize=7, color=COR_ROTULO[grupo])
        base += valores
    ax.set_xlim(0, 100)
    ax.set_xticks([])
    ax.spines["bottom"].set_visible(False)
    ax.tick_params(axis="y", length=0)
    ax.set_title(f"LTM até {rotulo(fim)}")
fig.legend(handles=[Patch(color=COR_GRUPO[g], label=ROTULO_GRUPO[g]) for g in GRUPOS_COMERCIAIS],
           loc="lower center", ncol=4, bbox_to_anchor=(0.5, -0.01))
fig.suptitle("Divisão das procedentes do comercial estrito por distribuidora (%)",
             x=0.01, ha="left", fontweight="bold", fontsize=11)
plt.tight_layout(rect=(0, 0.03, 1, 1))
plt.show()

lider = por_dx.xs(FIM, level="fim_janela").idxmax(axis=1)
print("grupo lider por distribuidora na ultima janela:")
print(lider.map(ROTULO_GRUPO).value_counts().to_string())
print("\ndistribuidoras em que o Faturamento nao lidera:", sorted(lider[lider != "faturamento"].index))

**Conclusão:** o Faturamento lidera em 23 distribuidoras. Em cinco, a Geração distribuída já é o maior grupo de procedentes: ENEL RJ (64%), COSERN (55%), COELBA (46%), Neoenergia PE (45%) e EQUATORIAL GO (43%). Em quatro, lidera Outras comerciais: ELEKTRO, EDP SP, ERO e EMR.

### Conclusão da P1, primeira parte

O consumidor reclama sobretudo de falta de energia. No comercial, reclama de faturamento e, cada vez mais, de geração distribuída, que já é o maior tema comercial em cinco distribuidoras.

## 2. Quem reduziu as reclamações comerciais (P2 e P1, segunda parte)

Para avaliar a qualidade do serviço comercial de uma distribuidora, o ponto de partida é a visão do todo: o total de reclamações comerciais procedentes, que responde diretamente à P2. Em seguida, o detalhe por grupo mostra onde a variação aconteceu, porque uma melhora no Faturamento e uma melhora na Geração distribuída têm causas e remédios diferentes.

#### Como ler os números

- **Indicador:** reclamações procedentes do nível 1 em 12 meses por mil unidades consumidoras (UCs), com o `NumCon` da base de continuidade no denominador. Dividir pelo número de UCs permite comparar distribuidoras de tamanhos diferentes.
- **Comparação:** LTM dez/24 contra LTM jun/26, com pontos intermediários em LTM jun/25 e LTM dez/25.
- **Posição:** 1 é a maior redução entre as 32.

#### Ranking do comercial estrito

Barras azuis são reduções do indicador; barras laranja, aumentos. A ordem é a do ranking.

In [ ]:
total = rk(RECORTE_TOTAL, "procedentes")
barras_variacao(total["var_pct"],
                "P2 - Comercial estrito: variação das procedentes por mil UCs",
                f"Variação entre o LTM {rotulo(INI)} e o LTM {rotulo(FIM)}")

ini_u, fim_u = indicador_universo(RECORTE_TOTAL, INI), indicador_universo(RECORTE_TOTAL, FIM)
print(f"universo: {ini_u:.2f} -> {fim_u:.2f} procedentes por mil UCs ({(fim_u / ini_u - 1) * 100:+.1f}%)")
print(f"distribuidoras com reducao: {(total['var_pct'] < 0).sum()} de {len(total)}")
# display(total[["posicao", "indicador_inicio", "indicador_fim", "var_pct", "var_abs",
#                "volume_inicio", "volume_fim"]].reset_index())

**Conclusão:** no conjunto das 32 distribuidoras, as procedentes do comercial estrito caíram de 6,20 para 5,67 por mil UCs (−8,5%). 20 distribuidoras reduziram o indicador e 12 aumentaram. ERO, Âmbar Amazonas e EMT lideram, com reduções próximas de 40%. No outro extremo, a COSERN mais que dobrou o indicador (+116%), seguida de CPFL-PAULISTA (+33%) e Neoenergia Brasília (+26%).

#### Trajetória nas quatro janelas

A comparação entre a primeira e a última janela não diz se a melhora foi contínua ou concentrada em um momento. As linhas mostram o indicador nas quatro janelas para as cinco maiores reduções e os cinco maiores aumentos. Para que distribuidoras de níveis diferentes caibam no mesmo gráfico, o valor da primeira janela vale 100: uma linha que termina em 70 reduziu 30%.

In [ ]:
serie = (univ[univ["recorte"] == RECORTE_TOTAL]
         .pivot(index="sig_agente", columns="fim_janela", values="procedentes_por_mil"))
indice = serie.div(serie[INI], axis=0) * 100
melhores, piores = total.index[:5], total.index[-5:]


def espacar(valores, distancia):
    """Spread end-label positions so that neighbours stay at least `distancia` apart."""
    ordem = sorted(valores, key=valores.get)
    posicao, anterior = {}, None
    for dx in ordem:
        alvo = valores[dx] if anterior is None else max(valores[dx], anterior + distancia)
        posicao[dx], anterior = alvo, alvo
    return posicao


fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
for ax, grupo, titulo in zip(axes, [melhores, piores], ["Cinco maiores reduções", "Cinco maiores aumentos"]):
    for dx in grupo:
        ax.plot(range(len(FINS)), indice.loc[dx, FINS], marker="o", markersize=4, linewidth=1.8)
    finais = {dx: indice.loc[dx, FIM] for dx in grupo}
    faixa = indice.loc[grupo, FINS].values
    alturas = espacar(finais, (faixa.max() - faixa.min()) * 0.07)
    for linha, dx in zip(ax.get_lines(), grupo):
        ax.annotate(f"{dx} {finais[dx]:.0f}", (len(FINS) - 1, finais[dx]), xytext=(len(FINS) - 0.85, alturas[dx]),
                    textcoords="data", va="center", fontsize=7.5, color=linha.get_color(),
                    arrowprops=dict(arrowstyle="-", color=linha.get_color(), linewidth=0.6))
    ax.set_xticks(range(len(FINS)), [f"LTM {rotulo(f)}" for f in FINS])
    ax.axhline(100, color=COR_EIXO, linewidth=0.8)
    grade_leve(ax, "y")
    ax.set_xlim(-0.2, len(FINS) + 0.9)
    ax.set_title(titulo)
    ax.set_ylabel("Índice (primeira janela = 100)")
fig.suptitle("Trajetória das procedentes por mil UCs no comercial estrito",
             x=0.01, ha="left", fontweight="bold", fontsize=11)
plt.tight_layout()
plt.show()

**Conclusão:** nas cinco maiores reduções, quase toda a queda aconteceu até dezembro de 2025; no último semestre, as variações são pequenas. Na COSERN, o salto concentra-se no último semestre: o índice vai de 121 para 216 entre o LTM dez/25 e o LTM jun/26.

#### De onde veio a variação: o universo

O indicador do comercial estrito é a soma dos quatro grupos, porque todos são divididos pelo mesmo número de UCs. Por isso, a variação do total pode ser repartida entre os grupos. Um exemplo com números redondos: uma distribuidora tinha 6 procedentes por mil UCs, sendo 3 de Faturamento, 1 de Pagamento, 1 de Geração distribuída e 1 de Outras, e passou a ter 5, sendo 2, 1, 1 e 1. A queda de 1 veio inteira do Faturamento.

O gráfico faz essa conta para o conjunto das 32 distribuidoras, somando reclamações e UCs de todas. Assim, cada distribuidora pesa conforme o seu número de clientes, como pesa para o consumidor brasileiro. Cada barra é a variação que um grupo provocou no indicador do universo, e a última é a soma delas, a variação total.

In [ ]:
passos = {g: indicador_universo(g, FIM) - indicador_universo(g, INI) for g in GRUPOS_COMERCIAIS}
residuo_u = abs(sum(passos.values()) - (fim_u - ini_u))
print(f"diferenca entre a soma dos grupos e o total do universo: {residuo_u:.4f} procedentes por mil UCs")

# Bars start at zero: each one is the change a group brought to the universe indicator
etapas = [passos[g] for g in GRUPOS_COMERCIAIS] + [fim_u - ini_u]
nomes_etapa = [ROTULO_GRUPO[g] for g in GRUPOS_COMERCIAIS] + ["Variação total"]
fig, ax = plt.subplots(figsize=(8, 3.6))
cores = [COR_MELHORA if d < 0 else COR_PIORA for d in etapas]
ax.bar(range(len(etapas)), etapas, color=cores, width=0.6)
ax.bar(len(etapas) - 1, etapas[-1], color="none", edgecolor="#2b2b29", linewidth=1.2, width=0.6)
margem = max(abs(d) for d in etapas) * 0.06
for i, d in enumerate(etapas):
    ax.text(i, d + (margem if d >= 0 else -margem), ("+" if d >= 0 else "") + br(d), ha="center",
            va="bottom" if d >= 0 else "top", fontsize=9)
ax.axhline(0, color=COR_EIXO, linewidth=0.8)
ax.set_xticks(range(len(etapas)), nomes_etapa)
amplitude = max(etapas + [0]) - min(etapas + [0])
ax.set_ylim(min(etapas + [0]) - amplitude * 0.2, max(etapas + [0]) + amplitude * 0.2)
ax.set_yticks([])
ax.spines["left"].set_visible(False)
ax.spines["bottom"].set_visible(False)
ax.set_title(f"P1 - Universo: contribuição de cada grupo (de {br(ini_u)} para {br(fim_u)} procedentes por mil UCs)",
             fontsize=10)
plt.tight_layout()
plt.show()

for g in GRUPOS_COMERCIAIS:
    print(f"{ROTULO_GRUPO[g]:<20} {passos[g]:+.2f} procedentes por mil UCs "
          f"({passos[g] / (fim_u - ini_u) * 100:.0f}% da variacao total)")

**Conclusão:** Faturamento (−0,40 procedentes por mil UCs), Outras comerciais (−0,25) e Pagamento (−0,14) caíram. A Geração distribuída subiu +0,27 e anulou cerca de metade dessa queda. Sem ela, o indicador teria caído de 6,20 para 5,41 por mil UCs (−12,7%), e não para 5,67.

#### De onde veio a variação: cada distribuidora

O gráfico faz a mesma conta para cada distribuidora. Cada cor é um grupo: segmentos à esquerda do zero são grupos que reduziram, à direita, grupos que aumentaram. O ponto preto é a variação total, a soma dos segmentos. O segmento mais longo mostra o grupo que mais contribuiu.

In [ ]:
delta = pd.DataFrame({g: valor(g, FIM, "procedentes_por_mil") - valor(g, INI, "procedentes_por_mil")
                      for g in GRUPOS_COMERCIAIS})
delta["total"] = valor(RECORTE_TOTAL, FIM, "procedentes_por_mil") - valor(RECORTE_TOTAL, INI, "procedentes_por_mil")
delta = delta.loc[total.index].sort_values("total")

residuo = (delta[GRUPOS_COMERCIAIS].sum(axis=1) - delta["total"]).abs().max()
print(f"maior diferenca entre a soma dos grupos e o total: {residuo:.4f} procedentes por mil UCs")

fig, ax = plt.subplots(figsize=(9, max(5, len(delta) * 0.26)))
pos_base = np.zeros(len(delta))
neg_base = np.zeros(len(delta))
for grupo in GRUPOS_COMERCIAIS:
    v = delta[grupo].values
    left = np.where(v >= 0, pos_base, neg_base)
    ax.barh(delta.index, v, left=left, height=0.7, color=COR_GRUPO[grupo],
            edgecolor="white", linewidth=0.8, label=ROTULO_GRUPO[grupo])
    pos_base += np.where(v >= 0, v, 0)
    neg_base += np.where(v < 0, v, 0)
ax.scatter(delta["total"], delta.index, color="black", s=16, zorder=3, label="Variação total")
ax.axvline(0, color=COR_EIXO, linewidth=0.8)
grade_leve(ax, "x")
ax.invert_yaxis()
ax.set_xlabel("Contribuição para a variação (procedentes por mil UCs)")
ax.legend(loc="upper right")
ax.set_title("P1 e P2 - Contribuição de cada grupo, por distribuidora")
plt.tight_layout()
plt.show()

**Conclusão:** na EMT, a maior redução absoluta, a queda veio sobretudo do Faturamento; na CEEE-D, de Outras comerciais. A ENEL RJ reduziu em todos os grupos, menos em Geração distribuída, onde as procedentes passaram de 843 para 7.629. Na COSERN, a maior parte do aumento veio da Geração distribuída.

#### Posição em cada ranking

Cada coluna é um ranking separado, e a posição 1 é a maior redução. Azul indica as primeiras posições; laranja, as últimas. Ler por linha mostra o perfil da distribuidora: uma linha toda azul é melhora generalizada; uma linha com cores misturadas mostra que a melhora do total veio de poucos grupos. A última coluna, de recebidas, antecipa a seção 3.

In [ ]:
posicoes = pd.DataFrame({
    "Total procedentes": rk(RECORTE_TOTAL, "procedentes")["posicao"],
    "Faturamento": rk("faturamento", "procedentes")["posicao"],
    "Pagamento": rk("pagamento", "procedentes")["posicao"],
    "Geração distribuída": rk("geracao_distribuida", "procedentes")["posicao"],
    "Outras comerciais": rk("outras_comerciais", "procedentes")["posicao"],
    "Total recebidas": rk(RECORTE_TOTAL, "recebidas")["posicao"],
}).sort_values("Total procedentes")

mapa = LinearSegmentedColormap.from_list("posicao", [COR_MELHORA, "#f4f3ef", COR_PIORA])
fig, ax = plt.subplots(figsize=(8.5, max(6, len(posicoes) * 0.26)))
ax.imshow(posicoes.values, cmap=mapa, aspect="auto", vmin=1, vmax=len(posicoes))
for i in range(posicoes.shape[0]):
    for j in range(posicoes.shape[1]):
        v = posicoes.iat[i, j]
        if pd.isna(v):
            # Company outside this ranking (no complaints in the initial window)
            ax.text(j, i, "fora", ha="center", va="center", fontsize=7, color=COR_TEXTO, style="italic")
            continue
        extremo = v <= 6 or v >= len(posicoes) - 5
        ax.text(j, i, f"{v:.0f}", ha="center", va="center", fontsize=7.5,
                color="white" if extremo else "#2b2b29")
ax.set_xticks(range(posicoes.shape[1]), [c.replace(" ", "\n", 1) for c in posicoes.columns], fontsize=8)
ax.xaxis.tick_top()
ax.set_yticks(range(len(posicoes)), posicoes.index, fontsize=8)
ax.tick_params(length=0)
for lado in ax.spines.values():
    lado.set_visible(False)
ax.set_title("Posição em cada ranking (1 = maior redução)", pad=28)
plt.tight_layout()
plt.show()

**Conclusão:** ERO está entre as oito primeiras em todos os rankings: melhora generalizada. A ENEL RJ é a primeira em Faturamento e em Outras comerciais, mas a 31ª em Geração distribuída. A COSERN é a última no total, em Faturamento e em Outras comerciais. A CEEE-D é a 7ª em procedentes, mas a 27ª em recebidas, caso que a seção 3 examina.

### Conclusão da P2

As reclamações comerciais procedentes caíram 8,5% por mil UCs, com 20 das 32 distribuidoras em queda, lideradas por ERO, Âmbar Amazonas e EMT. A redução veio do Faturamento, de Outras comerciais e do Pagamento; a Geração distribuída cresceu e anulou metade dela.

## 3. A procedência conta toda a história? (P3)

A procedência é classificada pela própria distribuidora. Uma queda nas procedentes pode vir de o cliente reclamar menos ou de a empresa reconhecer menos. As recebidas não dependem dessa classificação e oferecem um retrato do sentimento geral do cliente.

#### Ranking das recebidas

O gráfico repete o ranking da seção 2, agora com as recebidas por mil UCs. Barras azuis são reduções; laranja, aumentos.

In [ ]:
receb = rk(RECORTE_TOTAL, "recebidas")
barras_variacao(receb["var_pct"],
                "P3 - Comercial estrito: variação das recebidas por mil UCs",
                f"Variação entre o LTM {rotulo(INI)} e o LTM {rotulo(FIM)}")

universo_ini = univ[(univ["recorte"] == RECORTE_TOTAL) & (univ["fim_janela"] == INI) & univ["sig_agente"].isin(MEMBROS)]
universo_fim = univ[(univ["recorte"] == RECORTE_TOTAL) & (univ["fim_janela"] == FIM) & univ["sig_agente"].isin(MEMBROS)]
for nome, col in [("recebidas", "qtd_recebidas"), ("procedentes", "qtd_procedentes")]:
    a, b = universo_ini[col].sum(), universo_fim[col].sum()
    print(f"{nome:<12} universo: {a:>12,.0f} -> {b:>12,.0f} ({(b / a - 1) * 100:+.1f}%)")
ri, rf = indicador_universo(RECORTE_TOTAL, INI, "qtd_recebidas"), indicador_universo(RECORTE_TOTAL, FIM, "qtd_recebidas")
print(f"recebidas por mil UCs no universo: {ri:.2f} -> {rf:.2f} ({(rf / ri - 1) * 100:+.1f}%)")
print(f"distribuidoras com reducao das recebidas: {(receb['var_pct'] < 0).sum()} de {len(receb)}")

**Conclusão:** no universo, as recebidas por mil UCs caíram 9,1%, na mesma ordem das procedentes (−8,5%). No agregado, o cliente também passou a reclamar menos. Por distribuidora, 18 das 32 reduziram as recebidas; COSERN (+184%) e Neoenergia Brasília (+102%) têm as maiores altas.

#### Posição em procedentes contra posição em recebidas

Cada ponto é uma distribuidora. No eixo horizontal está a posição no ranking de procedentes; no vertical, a posição no ranking de recebidas. Sobre a linha tracejada, as duas posições coincidem.

- **Acima da linha (laranja):** a distribuidora está mais bem colocada em procedentes do que em recebidas. Reduziu mais o que ela própria reconhece do que aquilo de que o cliente reclama.
- **Abaixo da linha (azul):** o contrário. O cliente passou a reclamar bem menos, mas a redução das procedentes foi menor.

As cores marcam as distribuidoras com diferença de dez posições ou mais.

A **correlação de Spearman** resume a concordância entre os dois rankings em um número de −1 a +1: +1 significa ordens idênticas, 0 significa nenhuma relação e −1, ordens invertidas. Como referência usual, abaixo de 0,3 a relação é fraca, entre 0,3 e 0,7 é moderada e acima de 0,7 é forte. O **valor-p** é a chance de uma concordância desse tamanho aparecer por acaso, se os rankings não tivessem relação; abaixo de 0,05 (5%), a relação é considerada real.

In [ ]:
pos = pd.DataFrame({"procedentes": rk(RECORTE_TOTAL, "procedentes")["posicao"],
                    "recebidas": rk(RECORTE_TOTAL, "recebidas")["posicao"]})
pos["diferenca"] = pos["recebidas"] - pos["procedentes"]
n = len(pos)
destaque_pos = list(pos.index[pos["diferenca"].abs() >= 10])
cores_pos = pd.Series(COR_PONTO, index=pos.index)
cores_pos[pos["diferenca"] >= 10] = COR_PIORA
cores_pos[pos["diferenca"] <= -10] = COR_MELHORA
rho, p = spearmanr(pos["procedentes"], pos["recebidas"])

fig, ax = plt.subplots(figsize=(8, 7.5))
ax.plot([1, n], [1, n], color=COR_EIXO, linewidth=0.8, linestyle=(0, (4, 3)))
ax.scatter(pos["procedentes"], pos["recebidas"], c=cores_pos.tolist(), s=38, edgecolor="white", linewidth=0.8, zorder=3)
ax.set_xlim(0, n + 1)
ax.set_ylim(0, n + 1)
ax.set_xlabel("Posição em procedentes (1 = maior redução)")
ax.set_ylabel("Posição em recebidas (1 = maior redução)")
ax.set_title(f"P3 - Posição em procedentes e em recebidas (Spearman {br(rho)}, valor-p {'< 0,001' if p < 0.001 else br(p, 3)})")
rotular_pontos(ax, pos["procedentes"], pos["recebidas"], {dx: dx for dx in pos.index}, destaque_pos, cores_pos)
plt.tight_layout()
plt.show()

print(f"correlacao de Spearman entre as posicoes: {rho:.2f} (valor-p {p:.4f})")
# display(pos.reindex(pos["diferenca"].abs().sort_values(ascending=False).index).head(10).reset_index())

**Conclusão:** a correlação é de 0,67, com valor-p abaixo de 0,001: os dois rankings concordam na maior parte, mas não são iguais. CEEE-D (7ª em procedentes, 27ª em recebidas), Neoenergia PE (18ª e 30ª), ELEKTRO (10ª e 21ª) e EDP ES (6ª e 16ª) reduziram mais o que reconhecem do que aquilo de que o cliente reclama. EQUATORIAL GO (27ª e 11ª), EQUATORIAL PI (29ª e 17ª) e EMR (25ª e 15ª) fazem o caminho inverso.

#### Matriz recebidas × taxa de procedência

A matriz cruza duas variações de cada distribuidora. No eixo horizontal, a variação das recebidas por mil UCs: o cliente reclama mais ou menos. No eixo vertical, a variação da taxa de procedência (procedentes divididas por recebidas), em pontos percentuais: a distribuidora reconhece mais ou menos do que recebe. O corte é zero, e cada quadrante tem uma leitura, resumida no esquema abaixo.

Quando uma distribuidora fica muito distante das demais, a escala se estenderia só para ela e comprimiria as outras. Nesses casos, o ponto é desenhado na borda, em losango, com o valor real no rótulo. O critério é o do gráfico de caixa (boxplot): valor a mais de 1,5 vez a distância entre o primeiro e o terceiro quartis.

In [ ]:
ROTULO_P3 = {(False, False): "recebidas e taxa em queda",
             (False, True): "recebidas em queda, taxa em alta",
             (True, False): "recebidas em alta, taxa em queda",
             (True, True): "recebidas e taxa em alta"}
FUNDO_P3 = {(False, False): FUNDO_BOM, (False, True): FUNDO_NEUTRO,
            (True, False): FUNDO_ALERTA, (True, True): FUNDO_PIORA}

esquema_quadrantes(
    "Como ler a matriz recebidas × taxa de procedência",
    "Recebidas por mil UCs", "Taxa de procedência",
    {(False, False): "Melhora:\nmenos reclamações\ne menor reconhecimento",
     (False, True): "Menos reclamações,\ne as que chegam\nprocedem mais",
     (True, False): "Alerta:\no cliente reclama mais\ne a empresa reconhece menos",
     (True, True): "Piora reconhecida:\nmais reclamações\ne mais procedência"},
    FUNDO_P3)

Cada ponto é uma distribuidora, e o fundo de cada quadrante tem a cor do esquema. Em vermelho, as distribuidoras no quadrante de alerta; em azul, as que melhoraram nas duas dimensões.

In [ ]:
matriz = pd.DataFrame({
    "var_recebidas_pct": rk(RECORTE_TOTAL, "recebidas")["var_pct"],
    "var_taxa_pp": (valor(RECORTE_TOTAL, FIM, "taxa_procedencia")
                    - valor(RECORTE_TOTAL, INI, "taxa_procedencia")) * 100,
}).loc[MEMBROS].dropna()
matriz["quadrante_p3"] = [ROTULO_P3[(x >= 0, y >= 0)]
                          for x, y in zip(matriz["var_recebidas_pct"], matriz["var_taxa_pp"])]
alerta = matriz["quadrante_p3"] == ROTULO_P3[(True, False)]
melhora = matriz["quadrante_p3"] == ROTULO_P3[(False, False)]
cores_p3 = pd.Series(np.where(alerta, COR_ALERTA, np.where(melhora, COR_MELHORA, COR_PONTO)), index=matriz.index)

fig, ax = plt.subplots(figsize=(10, 7))
matriz_quadrantes(ax, matriz["var_recebidas_pct"], matriz["var_taxa_pp"], cores_p3,
                  list(matriz.index[alerta | melhora]), FUNDO_P3, unidade_y=" p.p.")
ax.set_xlabel(f"Variação das recebidas por mil UCs, LTM {rotulo(INI)} a LTM {rotulo(FIM)} (%)")
ax.set_ylabel("Variação da taxa de procedência (pontos percentuais)")
ax.set_title("P3 - Recebidas e taxa de procedência")
plt.tight_layout()
plt.show()

print(matriz["quadrante_p3"].value_counts().to_string())
# display(matriz.sort_values("var_recebidas_pct").round(1).reset_index())

**Conclusão:** 11 distribuidoras recebem mais reclamações e reconhecem uma parcela menor delas. Estão nesse quadrante de alerta as cinco distribuidoras do grupo Neoenergia (COELBA, COSERN, ELEKTRO, Neoenergia PE e Neoenergia Brasília), além de CEEE-D, COPEL-DIS, CPFL-PIRATINING, CPFL JAGUARI, EDP SP e LIGHT SESA. Na CEEE-D, as recebidas sobem 19% e a taxa de procedência cai 14,6 pontos. Os dados não permitem dizer se a causa é mais rigor na análise ou menos reconhecimento de reclamações fundamentadas. Oito distribuidoras melhoraram nas duas dimensões: Âmbar Amazonas, EMS, ERO, EMT, ELETROPAULO, ESS, EDP ES e EQUATORIAL MA.

#### Nível 1 × nível 2

O cliente que discorda da resposta do nível 1, ou cuja reclamação não foi resolvida, recorre à ouvidoria, o nível 2. Comparar a variação das recebidas nos dois níveis mostra se a redução no nível 1 foi acompanhada pela ouvidoria. A comparação usa o total do comercial estrito, em valores por mil UCs; por grupo ou por tipologia ela não é confiável (seção 6).

In [ ]:
ROTULO_N2 = {(False, False): "N1 e N2 em queda",
             (False, True): "N1 em queda, N2 em alta",
             (True, False): "N1 em alta, N2 em queda",
             (True, True): "N1 e N2 em alta"}
FUNDO_N2 = {(False, False): FUNDO_BOM, (False, True): FUNDO_ALERTA,
            (True, False): FUNDO_NEUTRO, (True, True): FUNDO_PIORA}

esquema_quadrantes(
    "Como ler a matriz nível 1 × nível 2",
    "Recebidas no nível 1 (atendimento da distribuidora)", "Recebidas no nível 2 (ouvidoria)",
    {(False, False): "Melhora consistente:\nmenos reclamações\nnos dois níveis",
     (False, True): "Alerta:\no nível 1 cai e a ouvidoria sobe;\npossível falta de solução\nno primeiro atendimento",
     (True, False): "Mais reclamações no nível 1,\nresolvidas sem chegar\nà ouvidoria",
     (True, True): "Piora nos dois níveis"},
    FUNDO_N2)

Em vermelho, as distribuidoras no quadrante de alerta; em azul, as que reduziram as reclamações nos dois níveis.

In [ ]:
def variacao_pct(coluna):
    ini, fim = valor(RECORTE_TOTAL, INI, coluna), valor(RECORTE_TOTAL, FIM, coluna)
    return (fim / ini - 1) * 100


niveis = pd.DataFrame({"var_n1_pct": variacao_pct("recebidas_por_mil"),
                       "var_n2_pct": variacao_pct("recebidas_nivel_2_por_mil")}).loc[MEMBROS].dropna()
niveis["quadrante_n2"] = [ROTULO_N2[(x >= 0, y >= 0)] for x, y in zip(niveis["var_n1_pct"], niveis["var_n2_pct"])]
migracao = niveis["quadrante_n2"] == ROTULO_N2[(False, True)]
consistente = niveis["quadrante_n2"] == ROTULO_N2[(False, False)]
cores_n2 = pd.Series(np.where(migracao, COR_ALERTA, np.where(consistente, COR_MELHORA, COR_PONTO)),
                     index=niveis.index)

fig, ax = plt.subplots(figsize=(10, 7))
matriz_quadrantes(ax, niveis["var_n1_pct"], niveis["var_n2_pct"], cores_n2,
                  list(niveis.index[migracao | consistente]), FUNDO_N2)
ax.set_xlabel("Variação das recebidas no nível 1 por mil UCs (%)")
ax.set_ylabel("Variação das recebidas no nível 2 por mil UCs (%)")
ax.set_title("P3 - Nível 1 e nível 2")
plt.tight_layout()
plt.show()

print(niveis["quadrante_n2"].value_counts().to_string())

escalada = pd.DataFrame({"escalada_inicio_pct": valor(RECORTE_TOTAL, INI, "taxa_escalada") * 100,
                         "escalada_fim_pct": valor(RECORTE_TOTAL, FIM, "taxa_escalada") * 100}).loc[MEMBROS]
escalada["variacao_pp"] = escalada["escalada_fim_pct"] - escalada["escalada_inicio_pct"]
tot_n1 = [universo_ini["qtd_recebidas_publicada"].sum(), universo_fim["qtd_recebidas_publicada"].sum()]
tot_n2 = [universo_ini["qtd_recebidas_nivel_2"].sum(), universo_fim["qtd_recebidas_nivel_2"].sum()]
print(f"\nescalada no universo: {tot_n2[0] / tot_n1[0] * 100:.1f}% -> {tot_n2[1] / tot_n1[1] * 100:.1f}%")
print(f"nivel 1 no universo: {tot_n1[0]:,.0f} -> {tot_n1[1]:,.0f} ({(tot_n1[1] / tot_n1[0] - 1) * 100:+.1f}%)")
print(f"nivel 2 no universo: {tot_n2[0]:,.0f} -> {tot_n2[1]:,.0f} ({(tot_n2[1] / tot_n2[0] - 1) * 100:+.1f}%)")
# display(escalada.join(niveis).sort_values("variacao_pp", ascending=False).round(1).reset_index())

**Conclusão:** no universo, o nível 1 caiu 7,1% e a ouvidoria subiu 13,6%. A taxa de escalada passou de 15,4 para 18,8 reclamações na ouvidoria para cada 100 no nível 1. 12 distribuidoras estão no quadrante de alerta, com o nível 1 em queda e a ouvidoria em alta; as maiores altas da escalada estão em ESE (de 29 para 67), ENEL RJ (de 26 para 62) e EMS (de 48 para 81). Só seis distribuidoras reduziram as reclamações nos dois níveis: ERO, Âmbar Amazonas, CEMIG-D, EQUATORIAL AL, EQUATORIAL PA e EQUATORIAL PI.

### Conclusão da P3

No agregado, a redução das procedentes vem acompanhada da redução das recebidas. Por distribuidora, não: das 20 que reduziram as procedentes, 15 convivem com mais reclamações recebidas ou com mais clientes recorrendo à ouvidoria. O ranking de procedentes, sozinho, não conta toda a história.

## 4. Técnico e comercial andam juntos? (P4)

A evolução real de uma distribuidora deve ser possível de observar tanto na frente técnica quanto na comercial. O que se tenta verificar aqui é se a melhora técnica de uma empresa ajuda a melhorar o atendimento comercial ou se um não impacta diretamente o outro.

#### Como ler o gráfico

Cada ponto é uma distribuidora. O eixo horizontal mostra a variação das procedentes de Qualidade, quase todas de falta de energia; o vertical, a variação das procedentes do comercial estrito. Em azul, as que melhoraram nos dois (quadrante inferior esquerdo); em laranja, as que pioraram nos dois (superior direito). Se os pontos se concentrarem nesses dois quadrantes, técnico e comercial andam juntos. O título traz a correlação de Spearman e o valor-p, explicados na seção 3.

In [ ]:
tec_com = pd.DataFrame({"var_qualidade_pct": rk("qualidade", "procedentes")["var_pct"],
                        "var_comercial_pct": rk(RECORTE_TOTAL, "procedentes")["var_pct"]}).loc[MEMBROS].dropna()

ROTULO_P4 = {(False, False): "melhora nos dois",
             (False, True): "melhora so no tecnico",
             (True, False): "melhora so no comercial",
             (True, True): "piora nos dois"}
tec_com["quadrante_p4"] = [ROTULO_P4[(x >= 0, y >= 0)]
                           for x, y in zip(tec_com["var_qualidade_pct"], tec_com["var_comercial_pct"])]
rho_p4, p_p4 = spearmanr(tec_com["var_qualidade_pct"], tec_com["var_comercial_pct"])
nos_dois = tec_com["quadrante_p4"] == ROTULO_P4[(False, False)]
piora_dois = tec_com["quadrante_p4"] == ROTULO_P4[(True, True)]
cores_p4 = pd.Series(np.where(nos_dois, COR_MELHORA, np.where(piora_dois, COR_PIORA, COR_PONTO)), index=tec_com.index)

fig, ax = plt.subplots(figsize=(10, 7))
matriz_quadrantes(ax, tec_com["var_qualidade_pct"], tec_com["var_comercial_pct"], cores_p4,
                  list(tec_com.index[nos_dois | piora_dois]),
                  {(False, False): FUNDO_BOM, (False, True): FUNDO_NEUTRO,
                   (True, False): FUNDO_NEUTRO, (True, True): FUNDO_PIORA})
ax.set_xlabel("Variação das procedentes de Qualidade por mil UCs (%)")
ax.set_ylabel("Variação das procedentes do comercial estrito por mil UCs (%)")
ax.set_title(f"P4 - Técnico e comercial (Spearman {br(rho_p4)}, valor-p {br(p_p4, 3)})")
plt.tight_layout()
plt.show()

print(f"Spearman: {rho_p4:.2f}   valor-p: {p_p4:.3f}   distribuidoras: {len(tec_com)}")
print(tec_com["quadrante_p4"].value_counts().to_string())
# display(tec_com.sort_values("quadrante_p4").round(1).reset_index())

**Conclusão:** a correlação é de 0,39, com valor-p de 0,025. A chance de ser acaso é pequena, mas o alinhamento é fraco a moderado: saber como uma distribuidora foi no técnico ajuda pouco a prever como foi no comercial. 18 distribuidoras andam na mesma direção nas duas frentes (11 melhoram nas duas, como CEEE-D, ERO e EMT; 7 pioram nas duas, como CPFL-PAULISTA e Neoenergia Brasília). As outras 14 andam em direções opostas: 9 melhoram só no comercial e 5 só no técnico, entre elas a COSERN, que reduz as reclamações de Qualidade em 13% e mais que dobra as comerciais.

### Conclusão da P4

Técnico e comercial evoluem, em boa parte, como frentes independentes. A melhora técnica não garante a melhora do atendimento comercial, e o consumidor precisa olhar as duas.

## 5. O técnico confirma a continuidade? (P5)

As reclamações de Qualidade são quase todas de falta de energia, e o DEC-FI e o FEC-FI medem a duração e a frequência dessas faltas. Se o pipeline estiver correto, a distribuidora que reduziu o DEC-FI e o FEC-FI deveria ter reduzido também essas reclamações. As duas medidas usam as mesmas janelas e o mesmo conjunto de distribuidoras.

#### Como ler os gráficos

Cada ponto é uma distribuidora. O eixo horizontal mostra a variação do indicador de continuidade; o vertical, a variação das reclamações de Qualidade. O fundo azul marca quem melhorou nos dois; o laranja, quem piorou nos dois. Se as duas medidas contam a mesma história, os pontos formam uma nuvem inclinada, de baixo à esquerda para cima à direita.

O título de cada gráfico traz dois números:

- **Spearman:** o grau de alinhamento, de −1 a +1. Positivo confirma a relação esperada; perto de zero, as duas medidas não conversam; negativo indicaria um erro de cálculo, porque não há razão para mais falta de energia gerar menos reclamação.
- **Valor-p:** a chance de um alinhamento desse tamanho aparecer por acaso, se as medidas não tivessem relação. Abaixo de 0,05 (5%), a relação é considerada real.

Em vermelho, as cinco distribuidoras com maior diferença entre a posição no ranking de Qualidade e a posição no ranking de continuidade.

In [ ]:
dec = ranking_cont[ranking_cont["indicador"] == "dec_fi"].set_index("sig_agente")
fec = ranking_cont[ranking_cont["indicador"] == "fec_fi"].set_index("sig_agente")
conf = pd.DataFrame({
    "posicao_qualidade": rk("qualidade", "procedentes")["posicao"],
    "var_qualidade_pct": rk("qualidade", "procedentes")["var_pct"],
    "posicao_dec_fi": dec["posicao"], "var_dec_fi_pct": dec["var_pct"],
    "posicao_fec_fi": fec["posicao"], "var_fec_fi_pct": fec["var_pct"],
}).loc[MEMBROS].dropna()
conf["diferenca_dec"] = conf["posicao_qualidade"] - conf["posicao_dec_fi"]
conf["diferenca_fec"] = conf["posicao_qualidade"] - conf["posicao_fec_fi"]
FUNDO_P5 = {(False, False): FUNDO_BOM, (False, True): FUNDO_NEUTRO,
            (True, False): FUNDO_NEUTRO, (True, True): FUNDO_PIORA}


def grafico_continuidade(col_var, col_dif, nome):
    """Quality complaints against one continuity indicator, with the five largest rank gaps in red."""
    rho, p = spearmanr(conf["var_qualidade_pct"], conf[col_var])
    divergentes = list(conf[col_dif].abs().sort_values(ascending=False).head(5).index)
    cores = pd.Series(np.where(conf.index.isin(divergentes), COR_ALERTA, COR_PONTO), index=conf.index)
    fig, ax = plt.subplots(figsize=(10, 7))
    matriz_quadrantes(ax, conf[col_var], conf["var_qualidade_pct"], cores, divergentes, FUNDO_P5)
    ax.set_xlabel(f"Variação do {nome}, LTM {rotulo(INI)} a LTM {rotulo(FIM)} (%)")
    ax.set_ylabel("Variação das reclamações de Qualidade (%)")
    ax.set_title(f"P5 - Reclamações de Qualidade e {nome} (Spearman {br(rho)}, valor-p {br(p, 3)})")
    plt.tight_layout()
    plt.show()
    mesma = ((conf[col_var] < 0) == (conf["var_qualidade_pct"] < 0)).sum()
    print(f"{nome}: Spearman {rho:.2f}, valor-p {p:.3f}")
    print(f"distribuidoras com a mesma direcao nas duas medidas: {mesma} de {len(conf)}")
    print(f"maiores divergencias de posicao: {divergentes}")


grafico_continuidade("var_dec_fi_pct", "diferenca_dec", "DEC-FI")

**Conclusão:** a correlação com o DEC-FI é de 0,48, com valor-p de 0,006: relação moderada, com chance de acaso inferior a 1%. As distribuidoras que reduziram a duração das interrupções tendem a ter reduzido também as reclamações de falta de energia, e 23 das 32 andam na mesma direção nas duas medidas. As divergências têm padrões diferentes. RGE SUL tem a maior redução do DEC-FI (−70%), mas as reclamações ficaram estáveis (+0,7%); a CEEE-D, atingida pelo mesmo evento de maio de 2024, reduziu os dois (−69% e −63%). EQUATORIAL GO e EQUATORIAL AL reduzem o DEC-FI e aumentam as reclamações; EMR e EMS fazem o oposto.

In [ ]:
grafico_continuidade("var_fec_fi_pct", "diferenca_fec", "FEC-FI")
# display(conf.sort_values("posicao_dec_fi").reset_index())

**Conclusão:** com o FEC-FI, a correlação é de 0,35, com valor-p de 0,050, exatamente no limite convencional de 5%. A relação existe, mas não pode ser afirmada com a mesma segurança da do DEC-FI. 21 das 32 distribuidoras andam na mesma direção nas duas medidas. EQUATORIAL GO, EQUATORIAL AL e EMR repetem as divergências do DEC-FI. A novidade é a ELETROPAULO, que reduz as reclamações de Qualidade em 25% enquanto o FEC-FI sobe 13%; a EMS repete o padrão, com FEC-FI em alta de 28%.

### Conclusão da P5

Sim, com ressalvas. O ranking de reclamações de Qualidade acompanha o de continuidade de forma moderada, com associação mais forte com a duração (DEC-FI) do que com a frequência (FEC-FI) das interrupções. As duas correlações são positivas, a direção esperada, o que não indica erro de cálculo. Eventos climáticos na janela inicial distorcem a comparação em algumas distribuidoras; as divergências não têm causa apurada neste trabalho.

## 6. Robustez

Quatro verificações mostram quanto as conclusões dependem de decisões de tratamento ou de limitações do dado. Nesta seção, as tabelas são a própria evidência e ficam visíveis.

#### Sensibilidade à imputação

Seis meses do comercial estrito foram imputados na Silver. A tabela mostra as distribuidoras cuja posição muda quando o ranking é calculado com os valores publicados.

In [ ]:
sens = rk(RECORTE_TOTAL, "procedentes")
mudam = sens[sens["variacao_posicao"] != 0][["posicao", "posicao_sem_imputacao", "variacao_posicao",
                                              "var_pct", "var_pct_sem_imputacao"]]
print(f"distribuidoras com posicao alterada pela imputacao: {len(mudam)} de {len(sens)}")
print(f"maior mudanca de posicao: {mudam['variacao_posicao'].abs().max():.0f} posicoes")
display(mudam.reset_index())

**Conclusão:** 9 das 32 distribuidoras mudam de posição, mas só duas por efeito direto: COELBA (da 21ª para a 17ª) e ELEKTRO (da 10ª para a 13ª). As demais se deslocam uma posição como consequência. Nenhuma conclusão das seções anteriores depende da imputação.

#### Volume baixo

Nos grupos de volume baixo por distribuidora, poucas dezenas de reclamações movem a variação percentual. A tabela mostra as posições extremas de Pagamento, Geração distribuída e Outras comerciais ao lado do volume absoluto.

In [ ]:
for grupo in ["pagamento", "geracao_distribuida", "outras_comerciais"]:
    tabela = rk(grupo, "procedentes")[["posicao", "var_pct", "volume_inicio", "volume_fim"]]
    print(f"{ROTULO_GRUPO[grupo]}: volume final mediano de {tabela['volume_fim'].median():,.0f} procedentes")
    display(pd.concat([tabela.head(3), tabela.tail(3)]).reset_index())

**Conclusão:** no Pagamento, o volume mediano é de 512 procedentes na última janela, e os extremos refletem isso: a Neoenergia Brasília aparece com +1.049% porque passou de 106 para 1.245 procedentes. Na Geração distribuída, o volume mediano é de 1.662, mas os extremos também são movimentos de volume: a EDP ES cai 93% (de 1.322 para 91) e a ENEL RJ sobe 794% (de 843 para 7.629). A LIGHT SESA não registra reclamação de Geração distribuída nas duas janelas e fica fora desse ranking pela regra do indicador inicial positivo. Nesses grupos, a posição deve ser lida junto com o volume.

#### Inconsistência entre os níveis 1 e 2

Por tipologia, há distribuidoras em que o nível 2 recebe mais reclamações do que o nível 1 inteiro, todos os meses da série. A tabela lista os maiores casos na última janela.

In [ ]:
inconsistentes = (fato.filter(F.col("ano_mes").isin(meses_fim))
    .join(universo_spark, "num_cnpj")
    .join(tipologia.filter("ind_comercial_estrito").select("cod_tipologia", "descricao"), "cod_tipologia")
    .groupBy("num_cnpj", "cod_tipologia", "descricao")
    .agg(F.sum(F.when(F.col("nivel") == 1, F.col("qtd_recebidas_publicada")).otherwise(0)).alias("recebidas_n1"),
         F.sum(F.when(F.col("nivel") == 2, F.col("qtd_recebidas_publicada")).otherwise(0)).alias("recebidas_n2"))
    .filter(F.col("recebidas_n2") > F.col("recebidas_n1"))
    .toPandas().merge(nomes, on="num_cnpj"))

print(f"combinacoes distribuidora x tipologia com N2 acima de N1: {len(inconsistentes)}")
print(f"distribuidoras envolvidas: {inconsistentes['sig_agente'].nunique()}")
print(f"reclamacoes de N2 nessas combinacoes: {inconsistentes['recebidas_n2'].sum():,.0f} "
      f"({inconsistentes['recebidas_n2'].sum() / tot_n2[1] * 100:.1f}% do N2 do comercial estrito)")
display(inconsistentes.sort_values("recebidas_n2", ascending=False)
        [["sig_agente", "cod_tipologia", "descricao", "recebidas_n1", "recebidas_n2"]].head(15))

**Conclusão:** há 523 combinações de distribuidora e tipologia em que a ouvidoria recebe mais reclamações do que o nível 1, nas 32 distribuidoras, somando 43% das reclamações de ouvidoria do comercial estrito. O caso mais expressivo é a ELETROPAULO em Solicitação não atendida ou atrasada: 1.007 reclamações no nível 1 e 8.041 na ouvidoria. A inconsistência confirma a decisão de comparar os níveis só no total, onde a ouvidoria é menor que o nível 1 em todas as distribuidoras.

#### Exclusões

A CELESC está fora de todos os rankings. A decisão e a evidência estão no `02_silver_complaints`, seção 11.

In [ ]:
display(spark.table(f"{SILVER}.exclusao_ranking_manifestacao").toPandas().merge(nomes, on="num_cnpj"))

### Conclusão da robustez

As conclusões não dependem da imputação. As leituras por grupo de volume baixo e a comparação entre níveis por tipologia são as mais frágeis, e por isso o trabalho se apoia no total do comercial estrito e lê cada posição ao lado do volume.

## 7. Discussão geral

O quadro reúne, por distribuidora, as respostas às cinco perguntas: posição no comercial estrito por procedentes e por recebidas, posição em cada grupo, quadrante nas duas matrizes da P3, e posição em Qualidade e no DEC-FI.

In [ ]:
quadro = (posicoes
    .join(matriz["quadrante_p3"])
    .join(niveis["quadrante_n2"])
    .join(conf[["posicao_qualidade", "posicao_dec_fi"]])
    .sort_values("Total procedentes"))
display(quadro.reset_index())

#### O que os números dizem sobre a qualidade percebida pelo consumidor

Entre o LTM dez/24 e o LTM jun/26, as reclamações comerciais procedentes caíram 8,5% por mil UCs no conjunto das 32 distribuidoras. As recebidas caíram 9,1%. A melhora veio do Faturamento, de Outras comerciais e do Pagamento. A Geração distribuída andou na direção contrária: sem ela, a queda teria sido de 12,7%.

O agregado, porém, é uma média de trajetórias muito diferentes:

1. **Melhora consistente.** ERO, Âmbar Amazonas e EMT reduzem as procedentes e as recebidas, e ficam nas primeiras posições em quase todos os rankings. ERO e EMT também melhoram no técnico.
2. **Melhora com ressalva.** Das 20 distribuidoras que reduziram as procedentes, 15 têm alguma ressalva. ENEL RJ, EMS, EDP ES, ESE e ETO reduzem as reclamações no nível 1, mas veem a ouvidoria crescer. CEEE-D, ELEKTRO e EDP SP reduzem as procedentes enquanto as recebidas sobem. Para essas empresas, a posição no ranking de procedentes superestima a melhora percebida pelo consumidor.
3. **Piora.** COSERN, CPFL-PAULISTA e Neoenergia Brasília ocupam as últimas posições. Nas cinco distribuidoras do grupo Neoenergia, as recebidas sobem e a taxa de procedência cai, o que coloca o grupo inteiro no quadrante de alerta.

A Geração distribuída é o tema comercial que mais cresce. Em cinco distribuidoras ela já é o maior grupo de reclamações procedentes, e na ENEL RJ e na COSERN ela explica a maior parte da piora. Como o indicador é dividido pelo total de UCs, parte desse crescimento acompanha a expansão da própria geração distribuída, e não só a qualidade do atendimento.

Técnico e comercial evoluem de forma apenas fracamente relacionada. Já as reclamações de Qualidade acompanham o DEC-FI de forma moderada, o que valida o pipeline: duas bases independentes da ANEEL contam a mesma história na direção esperada.

#### Limites

Os resultados são descritivos e comparam duas janelas. A imputação altera pouco o ranking, mas o volume baixo em Pagamento e a inconsistência entre níveis limitam a leitura por grupo e por tipologia. O indicador de Geração distribuída não separa adesão de qualidade. Eventos climáticos na janela inicial distorcem a comparação da continuidade no Rio Grande do Sul.

## Pendências documentadas

| Item | Situação | O que falta |
|---|---|---|
| Faixa de estabilidade nas matrizes | Corte em zero, sem faixa neutra | Definir a faixa com base estatística, por exemplo a dispersão mensal da própria distribuidora |
| Inferência sobre a série mensal | Resultados descritivos | Gráfico de controle tipo u, Mann-Kendall e bootstrap de posição sobre a série mensal |
| Eventos climáticos na base | Efeito de base apontado na P5 | Estudo em notebook próprio, com ranking que neutraliza os meses de evento, após a conclusão do trabalho |
| Inconsistência entre níveis | Comparação restrita ao total | Apurar a causa junto às distribuidoras ou à ANEEL |
| Denominador da Geração distribuída | Total de UCs | Incluir o número de unidades com geração distribuída, publicado pela ANEEL, para separar adesão de qualidade |

## Autoavaliação desta etapa

**O que esta etapa entregou.** Respondi às cinco perguntas com base nas quatro tabelas da Gold, sem nenhum cálculo de métrica fora delas. A exceção são as visões por grupo da REH e por bloco, que leem a Silver porque a Gold agrega por grupo de ranking. A análise mostrou que o ranking de procedentes, sozinho, superestima a melhora de parte das distribuidoras, e isso justificou ter mantido os rankings de recebidas e a comparação com a ouvidoria.

**O que mudou no caminho.**

- A Geração distribuída saiu de Outras comerciais e ganhou ranking próprio. A primeira rodada da análise mostrou que ela é o 2º grupo do comercial estrito e o que mais cresce, e que, dentro de Outras, fazia a piora da ENEL RJ e da COSERN parecer um problema de atendimento ou cadastro. A mudança voltou à Silver de referência e à Gold.
- A primeira versão da decomposição somava as variações de cada distribuidora, o que dava o mesmo peso a distribuidoras de portes muito diferentes. Corrigi para a decomposição do indicador do universo.
- O notebook foi reorganizado para leitura rápida: cada pergunta abre com o que será analisado e como ler o gráfico, cada gráfico é seguido da sua conclusão, e cada pergunta fecha com uma conclusão curta. As tabelas de apoio foram comentadas, com exceção das que são a própria evidência.
- Os gráficos foram refeitos para um público de gestão: esquemas para as matrizes de quadrantes, valores dentro das barras, todas as distribuidoras nomeadas, com destaque em cor para as citadas, e pontos extremos presos à borda, para que um único valor não comprima a escala das demais.
- A abertura por grupo da REH (2º nível) substituiu o top 10 de tipologias, que repetia rótulos genéricos como "Outros" e não mostrava o peso de cada grupo.

**O que ficou em aberto.**

- A inconsistência entre níveis é maior do que eu havia estimado: 43% das reclamações de ouvidoria do comercial estrito estão em tipologias em que a ouvidoria supera o nível 1. A causa segue não apurada.
- O indicador de Geração distribuída precisaria do número de unidades com geração distribuída no denominador para separar adesão de qualidade. Esse dado não está no pipeline. A LIGHT SESA, sem nenhuma reclamação de GD, também merece verificação.
- As divergências entre continuidade e reclamações de Qualidade (RGE SUL, EQUATORIAL GO, EQUATORIAL AL, EMR, EMS e ELETROPAULO) não têm explicação neste trabalho.
- O estudo dos eventos climáticos fica para depois da conclusão do trabalho.

**O que eu faria diferente.**

- Olharia a evolução de cada grupo ao longo das janelas antes de fixar os recortes. O Pareto de 2024 mostrava a Geração distribuída como grupo restrito; o crescimento dela só apareceu na série.
- Trataria a série mensal com inferência estatística, e não só duas janelas: gráfico de controle tipo u para separar oscilação normal de mudança real, teste de Mann-Kendall para tendência e bootstrap para o intervalo de cada posição no ranking.
- Definiria o corte dos quadrantes com uma faixa de estabilidade de base estatística, em vez do zero, para não classificar como melhora ou piora variações dentro da oscilação normal da distribuidora.
- Planejaria os visuais e a estrutura de leitura para o público final desde o início, em vez de refazê-los depois da primeira rodada.